# 05 Models — XGBoost — Men's

XGBoost is the consensus top performer for this competition (research). We use shallow trees with regularization and LOGO-CV for honest evaluation.

**Approach:**
- LOGO cross-validation (each season = one fold)
- Train on all other seasons, predict held-out season's tournament games
- Evaluate with Brier score (MSE of predicted probabilities vs actual outcomes)
- Train final model on all data for Stage 1/2 predictions
- Apply Platt scaling (isotonic regression) for calibration

**Inputs** (from S3 `04_preprocessing/mens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/xgboost/mens/`):
- `oof_predictions.parquet` — out-of-fold predictions for all training matchups
- `stage1_predictions.parquet` — predictions for Stage 1 grid
- `stage2_predictions.parquet` — predictions for Stage 2 grid
- `cv_results.parquet` — per-fold Brier scores

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except:
    !pip install 'xgboost>=2.0,<3.0'
    import xgboost as xgb

from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

print(f"XGBoost version: {xgb.__version__}")

XGBoost version: 2.1.4


#### Functions

In [ ]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

def brier_objective(predt, dtrain):
    """Custom Brier score objective for XGBoost.
    Gradient and hessian of Brier loss w.r.t. raw predictions (before sigmoid).
    """
    labels = dtrain.get_label()
    preds = 1.0 / (1.0 + np.exp(-predt))  # sigmoid
    
    # Gradient: d(Brier)/d(raw) = 2(p - y) * p(1-p)
    grad = 2.0 * (preds - labels) * preds * (1.0 - preds)
    
    # Hessian: d²(Brier)/d(raw)²
    hess = 2.0 * preds * (1.0 - preds) * (preds * (1.0 - preds) + (preds - labels) * (1.0 - 2.0 * preds))
    # Ensure hessian is positive for stability
    hess = np.maximum(hess, 1e-6)
    
    return grad, hess

def brier_eval(predt, dtrain):
    """Custom Brier score evaluation metric for XGBoost."""
    labels = dtrain.get_label()
    preds = 1.0 / (1.0 + np.exp(-predt))  # sigmoid
    brier = np.mean((preds - labels) ** 2)
    return 'brier', brier

#### Constants

In [ ]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "mens"
STAGE = "05_models/xgboost"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [ ]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

## 2. XGBoost Hyperparameters

Research recommends shallow trees (max depth 2–4) with regularization. We use conservative settings to avoid overfitting on the limited tournament game dataset.

In [ ]:
xgb_params = {
    'max_depth': 3,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'seed': 42,
    'verbosity': 0,
    'disable_default_eval_metric': True,
}

N_ROUNDS = 500
EARLY_STOPPING = 50

print("XGBoost parameters:")
for k, v in xgb_params.items():
    print(f"  {k}: {v}")
print(f"  n_rounds: {N_ROUNDS}")
print(f"  early_stopping: {EARLY_STOPPING}")

## 3. LOGO Cross-Validation

Train on all folds except one, predict the held-out fold. Collect out-of-fold predictions for all training matchups.

In [7]:
# Only use the original (non-flipped) rows for OOF evaluation
# The flipped rows are for training only
# Original rows have TeamA < TeamB (submission format)
train_original = train[train['TeamA'] < train['TeamB']].copy()

folds = sorted(train['Fold'].unique())
print(f"Total folds: {len(folds)}")
print(f"Original (non-flipped) matchups: {len(train_original)}")

Total folds: 40
Original (non-flipped) matchups: 2585


In [ ]:
oof_preds = []
cv_results = []

for fold in folds:
    # Train on all folds except current (use flip-doubled data for training)
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols]
    y_train = train.loc[train_mask, 'Label']
    X_val = train_original.loc[val_mask, feature_cols]
    y_val = train_original.loc[val_mask, 'Label']
    
    if len(X_val) == 0:
        continue
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=N_ROUNDS,
        obj=brier_objective,
        evals=[(dval, 'val')],
        custom_metric=brier_eval,
        early_stopping_rounds=EARLY_STOPPING,
        verbose_eval=False
    )
    
    # With custom objective, predict returns raw scores — apply sigmoid
    raw_preds = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
    preds = 1.0 / (1.0 + np.exp(-raw_preds))
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': model.best_iteration
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}, BestRound={model.best_iteration}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

In [9]:
# Overall CV Brier score
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"\nCV Results Summary:")
print(f"  Mean Brier: {cv_df['BrierScore'].mean():.4f}")
print(f"  Std Brier: {cv_df['BrierScore'].std():.4f}")
print(f"  Min Brier: {cv_df['BrierScore'].min():.4f} (Fold {cv_df.loc[cv_df['BrierScore'].idxmin(), 'Fold']})")
print(f"  Max Brier: {cv_df['BrierScore'].max():.4f} (Fold {cv_df.loc[cv_df['BrierScore'].idxmax(), 'Fold']})")


Overall OOF Brier Score: 0.1832
Stage 1 (2022-2025) Brier Score: 0.1917

CV Results Summary:
  Mean Brier: 0.1830
  Std Brier: 0.0182
  Min Brier: 0.1436 (Fold 2007)
  Max Brier: 0.2164 (Fold 2022)


## 4. Train Final Model

Train on all data for generating Stage 1 and Stage 2 predictions. Use the median best round from CV as the number of boosting rounds.

In [ ]:
final_rounds = int(cv_df['BestRound'].median())
print(f"Final model rounds: {final_rounds} (median of CV best rounds)")

X_all = train[feature_cols]
y_all = train['Label']

dtrain_all = xgb.DMatrix(X_all, label=y_all)
final_model = xgb.train(xgb_params, dtrain_all, num_boost_round=final_rounds, obj=brier_objective)

## 5. Calibration with Isotonic Regression

XGBoost probabilities often cluster around 0.5. We fit isotonic regression on OOF predictions to improve calibration, then apply it to final predictions.

In [11]:
# Fit calibrator on OOF predictions
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

# Check calibration improvement on OOF
oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])

print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")
print(f"Improvement: {overall_brier - calibrated_brier:+.4f}")

OOF Brier (raw): 0.1832
OOF Brier (calibrated): 0.1804
Improvement: +0.0028


## 6. Generate Predictions

In [ ]:
# Stage 1 predictions — apply sigmoid to raw scores
dstage1 = xgb.DMatrix(stage1[feature_cols])
raw_preds_s1 = final_model.predict(dstage1)
stage1['Pred_xgboost'] = 1.0 / (1.0 + np.exp(-raw_preds_s1))
stage1['Pred_xgboost_calibrated'] = calibrator.predict(stage1['Pred_xgboost'].values)

# Clip to avoid extreme probabilities
stage1['Pred_xgboost_calibrated'] = stage1['Pred_xgboost_calibrated'].clip(0.02, 0.98)

print(f"Stage 1 predictions: {stage1.shape}")
print(f"  Pred range: [{stage1['Pred_xgboost_calibrated'].min():.3f}, {stage1['Pred_xgboost_calibrated'].max():.3f}]")
print(f"  Pred mean: {stage1['Pred_xgboost_calibrated'].mean():.3f}")

# Evaluate on actual Stage 1 games
stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier_raw = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_xgboost'])
    s1_brier_cal = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_xgboost_calibrated'])
    print(f"\n  Stage 1 Brier (raw): {s1_brier_raw:.4f}")
    print(f"  Stage 1 Brier (calibrated): {s1_brier_cal:.4f}")

In [ ]:
# Stage 2 predictions — apply sigmoid to raw scores
dstage2 = xgb.DMatrix(stage2[feature_cols])
raw_preds_s2 = final_model.predict(dstage2)
stage2['Pred_xgboost'] = 1.0 / (1.0 + np.exp(-raw_preds_s2))
stage2['Pred_xgboost_calibrated'] = calibrator.predict(stage2['Pred_xgboost'].values)
stage2['Pred_xgboost_calibrated'] = stage2['Pred_xgboost_calibrated'].clip(0.02, 0.98)

print(f"Stage 2 predictions: {stage2.shape}")
print(f"  Pred range: [{stage2['Pred_xgboost_calibrated'].min():.3f}, {stage2['Pred_xgboost_calibrated'].max():.3f}]")
print(f"  Pred mean: {stage2['Pred_xgboost_calibrated'].mean():.3f}")

## 7. Feature Importance

In [14]:
importance = final_model.get_score(importance_type='gain')
imp_df = pd.DataFrame({'Feature': importance.keys(), 'Gain': importance.values()})
imp_df = imp_df.sort_values('Gain', ascending=False)

print("Feature Importance (gain):")
for _, row in imp_df.iterrows():
    print(f"  {row['Feature']:30s}: {row['Gain']:.4f}")

Feature Importance (gain):
  SeedDiff                      : 78.1512
  SeedB                         : 14.9829
  SeedA                         : 14.6463
  AvgPointDiffDiff              : 14.5669
  MORDiff                       : 11.3724
  WLKDiff                       : 11.3440
  IsPowerConfDiff               : 10.2902
  TopSystemsAvgRankDiff         : 9.4190
  AvgTODiff                     : 6.4604
  NetEffDiff                    : 6.1385
  POMDiff                       : 5.7931
  OppFG3PctDiff                 : 5.6734
  OffEffDiff                    : 5.5819
  FG3PctDiff                    : 5.5224
  SAGDiff                       : 5.3093
  FGPctDiff                     : 5.3016
  AvgDRDiff                     : 5.1760
  WinPctDiff                    : 5.1690
  DefEffDiff                    : 4.9808
  AvgStlDiff                    : 4.9197
  AvgBlkDiff                    : 4.9021
  AvgORDiff                     : 4.7784
  AvgPossDiff                   : 4.6320
  FTPctDiff            

## 8. Save Outputs

In [15]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_xgboost', 'Pred_xgboost_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_xgboost', 'Pred_xgboost_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/mens/oof_predictions.parquet ((2585, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/mens/stage1_predictions.parquet ((261013, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/mens/stage2_predictions.parquet ((66430, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/mens/cv_results.parquet ((40, 4))


## 9. Summary

In [16]:
print("=" * 60)
print("XGBOOST MODEL SUMMARY — MEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
if stage1_brier:
    print(f"Stage 1 Brier Score: {stage1_brier:.4f}")
print(f"\nCV Folds: {len(cv_df)}")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"Final model rounds: {final_rounds}")
print(f"\nTop 5 features (gain):")
for _, row in imp_df.head().iterrows():
    print(f"  {row['Feature']}: {row['Gain']:.4f}")

XGBOOST MODEL SUMMARY — MEN'S

OOF Brier Score (raw): 0.1832
OOF Brier Score (calibrated): 0.1804
Stage 1 Brier Score: 0.1917

CV Folds: 40
Mean Brier: 0.1830 +/- 0.0182
Final model rounds: 138

Top 5 features (gain):
  SeedDiff: 78.1512
  SeedB: 14.9829
  SeedA: 14.6463
  AvgPointDiffDiff: 14.5669
  MORDiff: 11.3724
